# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### 1. My lane as an ML task (type)

This is a **classification model whose output is deployed as a ranking/scoring tool** — not a
clean fit into a single box, and I'd rather say that than force it.

The underlying prediction is binary classification: for each page, is it currently declining
(yes/no)? That label is observable today (`trend_direction == "down"`). But the decision this
feeds (Week 1, section 2) isn't "flag every declining page" — it's "which ~50 pages does the
reviewer pick up this cycle, out of thousands of candidates." That turns the classifier's output
(a predicted probability of decline) into a **ranking score**: sort every page by that
probability, hand the reviewer the top of the list. `scripts/03_train_model.py` in this repo does
exactly this — trains a classifier on `is_declining_label`, then ranks by predicted probability
and scores the ranking with Precision@50, not plain accuracy.

So: classification model, evaluated and used as a ranking/scoring tool. Naming only
"classification" would hide why Precision@K (not ROC-AUC alone) is the metric that matches the
real decision; naming only "ranking" would hide that the score comes from a model trained the
classification way, on an observed label — not a learned pairwise ranker.

In [1]:
import os
from pathlib import Path

# make this runnable the same way from Colab, VS Code, or `jupyter nbconvert`:
# walk up from the current directory until we find the starter csv, and clone
# the repo as a last resort (same fallback the Week 1 notebook uses).
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

review_capacity_per_cycle = 50  # matches Precision@50 in outputs/model_report.md
n_pages = len(df)
n_declining = int((df["trend_direction"] == "down").sum())

print(f"Pages in this slice: {n_pages:,}")
print(f"Pages currently tagged 'down': {n_declining:,} ({n_declining / n_pages * 100:.1f}%)")
print(f"Reviewer capacity per cycle: {review_capacity_per_cycle}")
print(f"Declining candidates per reviewer slot: {n_declining / review_capacity_per_cycle:.0f}:1")

Working directory: C:\Users\Laptop\Documents\fly


Pages in this slice: 30,000
Pages currently tagged 'down': 16,262 (54.2%)
Reviewer capacity per cycle: 50
Declining candidates per reviewer slot: 325:1


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### 2. Target or proxy

The target I actually want — "does this page need a content review this cycle" — is never
directly observed anywhere in the data. Nobody labeled pages "needs review: yes/no."

The proxy I'd train on instead is `is_declining_label`, defined exactly as the pipeline defines
it: 1 when `trend_direction == "down"`, else 0. `trend_direction` is itself computed from
`trend_pct`, an **observed** quantity — the percent change in impressions from the previous
30-day window to the last 30-day window — not something I or a rule invented after the fact.

But it is still a proxy, and the gap matters: a page can lose impressions (proxy = 1) for reasons
a reviewer wouldn't act on — seasonality, a client pausing promotion — and a page can be worth
reviewing (thin content, stale, weak CTR) without yet showing an impression decline (proxy = 0).
The data dictionary calls this out directly: `trend_direction` and `trend_pct` are the label
source and must never leak into the model's features.

In [2]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("is_declining_label distribution (%):")
print(df["is_declining_label"].value_counts(normalize=True).mul(100).round(1))
print()

df[[
    "content_id", "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d", "is_declining_label",
]].head(8)

is_declining_label distribution (%):
is_declining_label
1    54.2
0    45.8
Name: proportion, dtype: float64



,content_id,trend_direction,trend_pct,impressions_last_30d,impressions_prev_30d,is_declining_label
0,content_304f48230142,down,-41.4,578,987,1
1,content_a1fb4e703a9e,down,-57.7,2501,5915,1
2,content_9aa793d4d895,down,-60.9,2382,6089,1
3,content_331d6c4de07b,stable,-13.8,3626,4206,0
4,content_d99b7a2d90ca,down,-34.7,4211,6452,1
5,content_d4084a4bc775,down,-38.9,617,1009,1
6,content_9a34b442b552,down,-92.3,1,13,1
7,content_a63219c6e95a,stable,0.6,636,632,0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### 3. Success metric

**Precision@50**: of the top 50 pages the score ranks highest, what fraction are actually
declining (`is_declining_label == 1`)? I'm matching the pipeline's own metric
(`outputs/model_report.md`) instead of inventing my own, because it lines up with the real
constraint — a reviewer works through roughly 50 pages a cycle, top-down, then stops. ROC-AUC or
accuracy would reward getting the *whole* ranking right, including the ~29,950 pages nobody will
ever look at. That's not the decision being made.

What "good" looks like, using numbers already computable from this repo's committed baseline
(`outputs/model_report.md`): the hand-weighted rule baseline (`baseline_refresh_score`, 4 signals
combined with fixed weights a person picked) reaches 0.240 Precision@50; the trained model in
this repo reaches 0.740 — 37 of the top 50 ranked pages are genuinely declining, versus 12 of 50
for the hand-tuned rule. I'd call this lane successful if a model clears the hand-rule baseline
by a wide margin, which the repo's own result already shows is achievable here.

In [3]:
import sys
sys.path.append("scripts")
from ml_utils import precision_at_k

y = df["is_declining_label"]

# naive single-signal scores -- computed today, on this data, no training yet
naive_scores = {
    "staleness only (days_since_last_update)": df["days_since_last_update"],
    "low CTR only (-ctr)": -df["ctr"],
    "worse position only (avg_position, 0 = no data excluded)": df["avg_position"].where(df["avg_position"] > 0, 0),
}

for name, score in naive_scores.items():
    p50 = precision_at_k(y, score, k=50)
    print(f"Precision@50 using [{name}]: {p50:.3f}")

print()
print("Reference, already computed in this repo (outputs/model_report.md):")
print("  baseline_rules (4 hand-weighted signals): 0.240")
print("  random_forest (learned weights):          0.740")

Precision@50 using [staleness only (days_since_last_update)]: 0.520
Precision@50 using [low CTR only (-ctr)]: 0.580


Precision@50 using [worse position only (avg_position, 0 = no data excluded)]: 0.140

Reference, already computed in this repo (outputs/model_report.md):
  baseline_rules (4 hand-weighted signals): 0.240
  random_forest (learned weights):          0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### 4. The unit of analysis, as a real dataframe

One row = **one content page**: a single published article/page belonging to one client,
summarized over a trailing 90-day window as of export time. `content_id` is the unique key;
`client_id` groups pages by which of the 32 clients they belong to. Nothing here is a query, a
search term, or a daily snapshot — each page's 90 days of activity are already rolled up into one
row.

In [4]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content_id: {df['content_id'].nunique():,}  (matches row count -> one row per page)")
print(f"Unique client_id:  {df['client_id'].nunique()}")
print()

df[[
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "avg_position", "trend_direction", "is_declining_label",
]].head(5)

Shape: 30,000 rows x 45 columns
Unique content_id: 30,000  (matches row count -> one row per page)
Unique client_id:  32



,content_id,client_id,content_type,content_age_days,impressions_90d,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,44.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### 5. Why ML beats a fixed rule here

The naive single-signal scores below are a check on my own assumption, not a confirmation of it —
and they didn't come out the way I expected. Staleness alone (0.520 Precision@50) and low CTR
alone (0.580) actually *beat* the repo's hand-weighted 4-signal rule (`baseline_refresh_score`,
0.240); only position alone (0.140) does worse. So the failure here isn't "no single signal
works." It's that a fixed rule has to decide, once, in advance, which signals to trust and how
much weight to give each — and the repo's rule mixes four reasonable signals with hand-picked
weights (40% visibility / 30% freshness / 25% position / 5% depth) that turn out to be worse than
just trusting one of two single, unweighted signals on their own. The weighting is the weak
point, not the choice of signals.

That's exactly what a fixed rule can't self-correct and a learned model can. The correlations
computed below show every individual signal is weak across the *whole* dataset (none above
|0.09|) — no signal is reliably strong everywhere, even though one can concentrate correctly at
the very top of a 30,000-row ranking, which is a different, narrower thing than being globally
predictive. Which signals matter, and by how much, plausibly differs across the data (a `keyword
article` and a `feedly article` don't go stale for the same reasons) in ways a single fixed weight
per signal can't track. A model that learns weights and interactions from the labeled data,
instead of having them picked once by a person, is what closes the gap from these single/hand-
weighted scores (0.14-0.58) to what the trained random forest reaches: 0.740. I would not have
predicted CTR alone would beat the official rule before running the cell below — which is itself
the argument for letting a model find the weighting instead of trusting my own hand-picked one.

In [5]:
signals = ["days_since_last_update", "ctr", "avg_position", "word_count", "engagement_rate"]
corr = df[signals].corrwith(df["is_declining_label"])

print("Correlation of individual signals with is_declining_label (none is strongly linear alone):")
print(corr.round(3))

Correlation of individual signals with is_declining_label (none is strongly linear alone):
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
word_count                0.090
engagement_rate          -0.013
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.